In [ ]:
!pip install folium

In [ ]:
# loading libraries
import math
import numpy as np
import numpy.linalg as nla
import pandas as pd # type: ignore
from matplotlib import pyplot as plt
import seaborn as sns

import folium

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# reading in the data

data = pd.read_csv("birdclef-2025/train.csv")

In [ ]:
# explore data
print("Data Shape:", data.shape)
print(f"Number of Unique Values for each Column:\n{data.nunique()}")

data.head()

In [ ]:
# mapping primary - scientific - common labels
mapped = data[["primary_label", "scientific_name", "common_name"]]

mapped = mapped.drop_duplicates()
mapped.index = range(len(mapped))
mapped

**Here, we can confirm that the primary labels are 1:1 mapped to scientific names and common names**

In [ ]:
# also looking into the species
spec_dat = pd.read_csv("birdclef-2025/taxonomy.csv")

In [ ]:
print("Shape of Species data:", spec_dat.shape)
print(f"Unique values:\n{spec_dat.nunique()}")
spec_dat.head()
# taking out the taxon_id since it's duplicated
spec_dat = spec_dat.drop("inat_taxon_id", axis = 1)

print(spec_dat["class_name"].value_counts())
spec_dat.head()
# we have class name for each primary label

In [ ]:
# merging class_name
full_data = pd.merge(data, spec_dat, on = ["primary_label", "scientific_name", "common_name"], how = "left")
full_data.head()

In [ ]:
full_data.dtypes

In [ ]:
full_data = full_data.replace(to_replace=r"\[''\]", value=pd.NA, regex=True)
full_data.isna().sum()

In [ ]:
pd.set_option('display.max_rows', None)

full_data["type"].value_counts()

In [ ]:
# song/canto call/whistle/chamado; uncertain/?/no idea/aberrant; sound/noise/sonido/wing/drum/rattle; flight call; alarm call/alarm;
sound_categories = {
    "song/canto": {"song", "duet", "canto"},
    "call": {"call", "llamado"},
    "uncertain": {"uncertain", "?", "no idea", "aberrant"},
    "mating/groups": {"lek", "communal"},
    "hatching": {"hatch"},
    "immitation": {"immitation", "mimic"},
    "noise/drum": {"drumming", "rattle", "noise", "wing", "sound", "sonido", "tamborilar", "human", "dog", "cow", "frog"}
}

In [ ]:
import ast

sound = full_data[full_data["type"].notna()][["type", "class_name"]]
sound = pd.DataFrame(sound)
sound["type_list"] = sound["type"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])


def parse_label(label):
    label = label.strip().lower()
    if label.endswith("?"):
        return label[:-1].strip(), True
    return label, False

def parse_labels_list(label_list):
    parsed = [parse_label(lbl) for lbl in label_list]
    clean_labels = [lbl for lbl, _ in parsed]
    is_uncertain = any(flag for _, flag in parsed)
    return clean_labels, is_uncertain

sound[["clean_labels", "has_uncertain"]] = sound["type_list"].apply(
    lambda labels: pd.Series(parse_labels_list(labels))
)

sound.head()

In [ ]:
def assign_categories(label_str, category_dict):
    label = label_str.lower()
    matched_categories = []
    for category, keywords in category_dict.items():
        if any(keyword in label for keyword in keywords):
            matched_categories.append(category)
    return matched_categories if matched_categories else ["other"]

sound["category"] = sound["type"].astype(str).apply(lambda x: assign_categories(x, sound_categories))


In [ ]:
sound.head(30)

In [ ]:
for category, keywords in sound_categories.items():
    sound[category] = sound["clean_labels"].apply(
        lambda labels: any(
            any(keyword in label for keyword in keywords)
            for label in labels
        )
    )

In [ ]:
sound.head()

**Distribution EDA**

In [ ]:
full_data.columns

# full_data["class_name"].value_counts()

In [ ]:
plt.figure(figsize = (8, 5))

plt.hist(full_data["class_name"], color='olive', edgecolor='black')
plt.xlabel("Classes")
plt.ylabel("Counts")
plt.title("Distribution of Classes")
plt.show()


In [ ]:
insecta = full_data[full_data["class_name"] == "Insecta"]
amphibia = full_data[full_data["class_name"] == "Amphibia"]
mammalia = full_data[full_data["class_name"] == "Mammalia"]
aves = full_data[full_data["class_name"] == "Aves"]

insecta = insecta.sort_values("primary_label")
amphibia = amphibia.sort_values("primary_label")
mammalia = mammalia.sort_values("primary_label")
aves = aves.sort_values("primary_label")

In [ ]:
insecta_counts = insecta["primary_label"].value_counts()

plt.figure(figsize = (10, 6))
insecta_counts.plot(kind = "barh", color = "lightgrey", edgecolor = "black")
# plt.hist(aves["primary_label"], bins = 146, color = "skyblue", edgecolor = "black", orientation = "horizontal")
plt.xlabel("Counts")
plt.ylabel("Primary Labels")
plt.title("Counts for Primary Labels - Insecta")
plt.show()

In [ ]:
amphibia_counts = amphibia["primary_label"].value_counts()

plt.figure(figsize = (10, 6))
amphibia_counts.plot(kind = "barh", color = "skyblue", edgecolor = "black")
plt.xlabel("Counts")
plt.ylabel("Primary Labels")
plt.title("Counts for Primary Labels - Amphibia")
plt.show()

In [ ]:
mammalia_counts = mammalia["primary_label"].value_counts()

plt.figure(figsize = (10, 6))
mammalia_counts.plot(kind = "barh", color = "lightyellow", edgecolor = "black")
plt.xlabel("Counts")
plt.ylabel("Primary Labels")
plt.title("Counts for Primary Labels - Mammalia")
plt.show()

In [ ]:
aves.nunique()

aves_counts = aves["primary_label"].value_counts()

plt.figure(figsize = (10, 25))
aves_counts.plot(kind = "barh", color = "orange", edgecolor = "black")
# plt.hist(aves["primary_label"], bins = 146, color = "skyblue", edgecolor = "black", orientation = "horizontal")
plt.xlabel("Counts")
plt.ylabel("Primary Labels")
plt.title("Counts for Primary Labels - Aves")
plt.show()

In [ ]:
song_cat = ["song/canto", "call", "uncertain", "mating/groups", "hatching", "immitation", "noise/drum"]

counts = sound[song_cat].sum()

# Plot
plt.figure(figsize=(10, 6))
counts.plot(kind='bar', color='skyblue', edgecolor = "black")
plt.title('Counts of Each Sound Category')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
insecta_sound = sound[sound["class_name"] == "Insecta"]
amphibia_sound = sound[sound["class_name"] == "Amphibia"]
mammalia_sound = sound[sound["class_name"] == "Mammalia"]
aves_sound = sound[sound["class_name"] == "Aves"]

# shapes
print("Insecta sounds:", insecta_sound.shape)
print("Amphibia_sounds:", amphibia_sound.shape)
print("Mammalia sounds:", mammalia_sound.shape)
print("Aves sounds:", aves_sound.shape)

In [ ]:
amphibia_counts = amphibia_sound[song_cat].sum()
mammalia_counts = mammalia_sound[song_cat].sum()
aves_counts = aves_sound[song_cat].sum()

fig, axes = plt.subplots(1, 3, figsize = (20, 6))

amphibia_counts.plot(kind = "bar", color = "olive", edgecolor = "black", ax = axes[0])
axes[0].set_title("Amphibia")

mammalia_counts.plot(kind = "bar", color = "lightgrey", edgecolor = "black", ax = axes[1])
axes[1].set_title("Mammalia")

aves_counts.plot(kind = "bar", color = "skyblue", edgecolor = "black", ax = axes[2])
axes[2].set_title("Aves")

fig.suptitle("Distribution of Types of Sounds for Each Class", x = .25, fontsize = 15)
fig.supylabel("Counts", x = .1)
fig.supxlabel("Types of Sound", y = -.15)

plt.show()


**Map for visualizing distribution of data points**

In [ ]:
loc_data = full_data[full_data["longitude"].notna() & full_data["latitude"].notna()]


center_lat = loc_data['latitude'].mean()
center_lon = loc_data['longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=4)

In [ ]:
for _, row in loc_data.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,  # Size of the circle
        color='blue',  # Border color
        fill=True,
        fill_color='orange',  # Fill color
        fill_opacity=0.7    
        ).add_to(m)

In [ ]:
m
# this is sooooo slow!

In [ ]:
insecta_loc = loc_data[loc_data["class_name"] == "Insecta"]

i_center_lat = insecta_loc['latitude'].mean()
i_center_lon = insecta_loc['longitude'].mean()
insecta_m = folium.Map(location=[i_center_lat, i_center_lon], zoom_start=4)

insecta_loc.head()

In [ ]:
for _, row in insecta_loc.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,  # Size of the circle
        color='blue',  # Border color
        fill=True,
        fill_color='orange',  # Fill color
        fill_opacity=0.7    
        ).add_to(insecta_m)

In [ ]:
insecta_m

In [ ]:
amphibia_loc = loc_data[loc_data["class_name"] == "Amphibia"]

am_center_lat = amphibia_loc['latitude'].mean()
am_center_lon = amphibia_loc['longitude'].mean()
amphibia_m = folium.Map(location=[am_center_lat, am_center_lon], zoom_start=4)

In [ ]:
for _, row in amphibia_loc.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,  # Size of the circle
        color='blue',  # Border color
        fill=True,
        fill_color='orange',  # Fill color
        fill_opacity=0.7    
        ).add_to(amphibia_m)

In [ ]:
amphibia_m

In [ ]:
mammalia_loc = loc_data[loc_data["class_name"] == "Mammalia"]

m_center_lat = mammalia_loc['latitude'].mean()
m_center_lon = mammalia_loc['longitude'].mean()
mammalia_m = folium.Map(location=[m_center_lat, m_center_lon], zoom_start=4)

In [ ]:
for _, row in mammalia_loc.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,  # Size of the circle
        color='blue',  # Border color
        fill=True,
        fill_color='orange',  # Fill color
        fill_opacity=0.7    
        ).add_to(mammalia_m)

In [ ]:
mammalia_m

In [ ]:
aves_loc = loc_data[loc_data["class_name"] == "Aves"]

av_center_lat = aves_loc['latitude'].mean()
av_center_lon = aves_loc['longitude'].mean()
aves_m = folium.Map(location=[av_center_lat, av_center_lon], zoom_start=4)

In [ ]:
for _, row in aves_loc.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,  # Size of the circle
        color='blue',  # Border color
        fill=True,
        fill_color='orange',  # Fill color
        fill_opacity=0.7    
        ).add_to(aves_m)

In [ ]:
aves_m